In [1]:
from pathlib import Path
import gcamreader
import os
import pandas as pd
import numpy as np
from utils import convert_to_mt
import plotly.graph_objects as go
import plotly.io as pio

In [2]:
def to_Mt(row):
    val, unit = row['value'], row['Units']
    if unit == 'Tg':
        return val
    elif unit == 'Gg':
        return val * 1e-3
    elif unit == 'MTC':
        return val * (44.009 / 12.011)
    else:
        raise ValueError(f"Unknown unit: {unit}")

# AR5 100-yr GWP
GWP_AR5 = {
    'CO2':    1,
    'CH4': 28,  # Methane
    'CH4_AGR': 28,  # Methane from Agriculture
    'CH4_AWB': 28,  # Methane from Agricultural Waste Burning
    'N2O': 265,  # Nitrous Oxide
    'N2O_AGR': 265,  # Nitrous Oxide from Agriculture
    'N2O_AWB': 265,  # Nitrous Oxide from Agricultural Waste Burning
    'HFC125': 3500,
    'HFC134a':1430,
    'HFC143a':4470,
    'HFC23':  14800,
    'HFC32':  675,
    'HFC43':  1500,
    'HFC227ea':3220,
    'HFC236fa':9810,
    'SF6':    23500,
    'C2F6':   12200,
    'CF4':    6630,
}

In [3]:
dfCO2Map = pd.read_csv("./extdata/gcamreport/CO2_tech_map.csv", skiprows=[0])
dfCO2Map.head()

,sector,subsector,technology,var1,var2,var3,var4,var5,var6,var7,var8,var9,unit_conv
0,airCO2,airCO2,airCO2,Emissions|CO2,Emissions|CO2|Other Capture and Removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.666667
1,CO2 removal,dac,hightemp DAC NG,Emissions|CO2,Emissions|CO2|Other Capture and Removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.666667
2,CO2 removal,dac,hightemp DAC elec,Emissions|CO2,Emissions|CO2|Other Capture and Removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.666667
3,CO2 removal,dac,lowtemp DAC heatpump,Emissions|CO2,Emissions|CO2|Other Capture and Removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.666667
4,agricultural energy use,mobile,refined liquids,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Demand,Emissions|CO2|Energy|Demand|AFOFI,Emissions|CO2|Energy|Demand|Residential and Co...,NaN,NaN,NaN,3.666667


In [4]:
dfNonCO2Map = pd.read_csv("./extdata/gcamreport/nonCO2_emissions_sector_map.csv", skiprows=[0])
dfNonCO2Map

,sector,subsector,ghg,var1,var2,var3,var4,var5,var6,var7,var8,unit_conv
0,agricultural energy use,NaN,BC,Emissions|BC,Emissions|BC|Energy,Emissions|BC|Energy|Demand,Emissions|BC|Energy|Demand|AFOFI,NaN,NaN,Emissions|BC|Energy|Demand|Residential and Com...,Emissions|BC|Energy and Industrial Processes,1.0
1,agricultural energy use,NaN,CH4,Emissions|CH4,Emissions|CH4|Energy,Emissions|BC|Energy|Demand,Emissions|CH4|Energy|Demand|AFOFI,NaN,NaN,Emissions|CH4|Energy|Demand|Residential and Co...,Emissions|CH4|Energy and Industrial Processes,1.0
2,agricultural energy use,NaN,CO,Emissions|CO,Emissions|CO|Energy,Emissions|BC|Energy|Demand,Emissions|CO|Energy|Demand|AFOFI,NaN,NaN,Emissions|CO|Energy|Demand|Residential and Com...,Emissions|CO|Energy and Industrial Processes,1.0
3,agricultural energy use,NaN,N2O,Emissions|N2O,Emissions|N2O|Energy,Emissions|N2O|Energy|Demand,Emissions|N2O|Energy|Demand|AFOFI,NaN,NaN,Emissions|N2O|Energy|Demand|Residential and Co...,Emissions|N2O|Energy and Industrial Processes,1000.0
4,agricultural energy use,NaN,NH3,Emissions|NH3,Emissions|NH3|Energy,Emissions|NH3|Energy|Demand,Emissions|NH3|Energy|Demand|AFOFI,NaN,NaN,Emissions|NH3|Energy|Demand|Residential and Co...,Emissions|NH3|Energy and Industrial Processes,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...
918,urban processes,NaN,OC,Emissions|OC,Emissions|OC|Other,NaN,NaN,NaN,NaN,NaN,NaN,1.0
919,urban processes,NaN,SO2_1,Emissions|Sulfur,Emissions|Sulfur|Other,NaN,NaN,NaN,NaN,NaN,NaN,1.0
920,urban processes,NaN,SO2_2,Emissions|Sulfur,Emissions|Sulfur|Other,NaN,NaN,NaN,NaN,NaN,NaN,1.0
921,urban processes,NaN,SO2_3,Emissions|Sulfur,Emissions|Sulfur|Other,NaN,NaN,NaN,NaN,NaN,NaN,1.0


In [5]:
proj_path = Path("/data/project/tae/gcam-core")
xml_path = proj_path / "input" / "gcamdata" / "xml"
db_path = proj_path / "output"

In [6]:
dbpath = "../output/"  # relative to current working directory
dbfile = "database_basexdb_korea_2035_20250721_7"
conn = gcamreader.LocalDBConn(dbpath, dbfile)
queries = gcamreader.parse_batch_query(os.path.join('..', 'output', 'queries','Main_queries.xml'))

Database scenarios: Current-Policy, Current-Policy, Enhanced-Ambition


In [7]:
scenarios = list(conn.listScenariosInDB()['name'])
scenarios

['Current-Policy', 'Current-Policy', 'Enhanced-Ambition']

In [8]:
scenarios = ['Current-Policy', 'Enhanced-Ambition']

In [9]:
for i, q in enumerate(queries):
    print(i, q.title)

0 primary energy consumption by region (avg fossil efficiency)
1 primary energy consumption by region (direct equivalent)
2 primary energy consumption with CCS by region (direct equivalent)
3 resource production
4 resource production by tech and vintage
5 resource supply curves
6 regional primary energy prices
7 elec gen by region (incl CHP)
8 elec gen by subsector
9 elec gen by gen tech
10 elec gen by gen tech and cooling tech
11 elec gen by gen tech and cooling tech and vintage
12 elec gen by gen tech and cooling tech (new)
13 elec energy input by subsector
14 elec energy input by elec gen tech
15 elec energy input by elec gen tech and cooling tech
16 elec prices by sector
17 elec gen costs by subsector
18 elec gen costs by tech
19 elec gen costs by cooling tech
20 elec share-weights by subsector
21 elec share-weights by tech
22 elec share-weights by cooling tech
23 elec td inputs and outputs
24 cogeneration by region
25 elec consumption by demand sector
26 elec sector water withdraw

In [10]:
q = queries[195]
print(q.title)
df = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
df['scenario'] = df['scenario'].str.split(',').str[0]
df

ag production by crop type


,Units,scenario,region,sector,output,Year,value
0,EJ,Current-Policy,South Korea,biomass,biomass,2025,0.014657
1,EJ,Current-Policy,South Korea,biomass,biomass,2030,0.036043
2,EJ,Current-Policy,South Korea,biomass,biomass,2035,0.068101
3,EJ,Enhanced-Ambition,South Korea,biomass,biomass,2025,0.014617
4,EJ,Enhanced-Ambition,South Korea,biomass,biomass,2030,0.054879
...,...,...,...,...,...,...,...
289,billion m3,Enhanced-Ambition,South Korea,Forest,Forest,2015,0.004540
290,billion m3,Enhanced-Ambition,South Korea,Forest,Forest,2020,0.004670
291,billion m3,Enhanced-Ambition,South Korea,Forest,Forest,2025,0.004700
292,billion m3,Enhanced-Ambition,South Korea,Forest,Forest,2030,0.004746


In [11]:
df['sector'].unique()

array(['biomass', 'Corn', 'FiberCrop', 'FodderGrass', 'Fruits', 'Legumes',
       'MiscCrop', 'NutsSeeds', 'OilCrop', 'OtherGrain', 'Pasture',
       'Rice', 'RootTuber', 'Soybean', 'Vegetables', 'Wheat', 'Forest'],
      dtype=object)

In [12]:
df[(df['sector'] == 'Rice')]

,Units,scenario,region,sector,output,Year,value
96,Mt,Current-Policy,South Korea,Rice,Rice,1975,6.821400
97,Mt,Current-Policy,South Korea,Rice,Rice,1990,7.726534
98,Mt,Current-Policy,South Korea,Rice,Rice,2005,6.318193
99,Mt,Current-Policy,South Korea,Rice,Rice,2010,5.960649
100,Mt,Current-Policy,South Korea,Rice,Rice,2015,5.590278
101,Mt,Current-Policy,South Korea,Rice,Rice,2020,5.835997
102,Mt,Current-Policy,South Korea,Rice,Rice,2025,6.220256
103,Mt,Current-Policy,South Korea,Rice,Rice,2030,6.701762
104,Mt,Current-Policy,South Korea,Rice,Rice,2035,6.865262
231,Mt,Enhanced-Ambition,South Korea,Rice,Rice,1975,6.821400


In [13]:
q = queries[262]
print(q.title)
dfCO2 = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
dfCO2['scenario'] = dfCO2['scenario'].str.split(',').str[0]
dfCO2['sector'] = dfCO2['sector'].str.replace(r'_d(?:[1-9]|10)$', '', regex=True)
dfCO2['GHG'] = 'CO2'

CO2 emissions by sector (no bio) (excluding resource production)


In [14]:
q = queries[272]
print(q.title)
dfNonCO2 = conn.runQuery(q, scenarios=scenarios, regions=['South Korea'])
dfNonCO2['scenario'] = dfNonCO2['scenario'].str.split(',').str[0]
dfNonCO2['sector'] = dfNonCO2['sector'].str.replace(r'_d(?:[1-9]|10)$', '', regex=True)
dfNonCO2 = dfNonCO2[(dfNonCO2['GHG'].isin(GWP_AR5.keys()))]
dfNonCO2

nonCO2 emissions by subsector (excluding resource production)


,Units,scenario,region,sector,subsector,GHG,Year,value
0,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2005,0.253100
1,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2010,0.629604
2,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2015,0.717899
3,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2020,0.796548
4,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2025,0.783772
...,...,...,...,...,...,...,...,...
23044,Tg,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,N2O,2015,0.000123
23045,Tg,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,N2O,2020,0.000124
23046,Tg,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,N2O,2025,0.000122
23047,Tg,Enhanced-Ambition,South Korea,waste biomass for paper,biomass,N2O,2030,0.000121


In [15]:
mask1 = ((dfNonCO2['sector'] == 'UnmanagedLand') & (dfNonCO2['subsector'].isin(['ForestFire', 'GrasslandFires'])))
mask2 = ((dfNonCO2['sector'] == 'urban processes') & (dfNonCO2['subsector'].isin(['landfills', 'wastewater', 'waste_incineration'])))

dfNonCO2_1 = dfNonCO2[~(mask1 | mask2)]
dfNonCO2_2 = dfNonCO2[mask1 | mask2]

print(dfNonCO2_1.shape, dfNonCO2_2.shape)

(6151, 8) (90, 8)


In [16]:
for sec in dfCO2['sector'].unique():
    if sec not in dfCO2Map['sector'].unique():
        print(sec)

electricity


In [17]:
for sec in dfNonCO2['sector'].unique():
    if sec not in dfNonCO2Map['sector'].unique():
        print(sec)

In [18]:
dfNonCO2[(dfNonCO2['sector'] == 'chemical feedstocks')]

,Units,scenario,region,sector,subsector,GHG,Year,value


In [19]:
dfCO2Map.head()

,sector,subsector,technology,var1,var2,var3,var4,var5,var6,var7,var8,var9,unit_conv
0,airCO2,airCO2,airCO2,Emissions|CO2,Emissions|CO2|Other Capture and Removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.666667
1,CO2 removal,dac,hightemp DAC NG,Emissions|CO2,Emissions|CO2|Other Capture and Removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.666667
2,CO2 removal,dac,hightemp DAC elec,Emissions|CO2,Emissions|CO2|Other Capture and Removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.666667
3,CO2 removal,dac,lowtemp DAC heatpump,Emissions|CO2,Emissions|CO2|Other Capture and Removal,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.666667
4,agricultural energy use,mobile,refined liquids,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Demand,Emissions|CO2|Energy|Demand|AFOFI,Emissions|CO2|Energy|Demand|Residential and Co...,NaN,NaN,NaN,3.666667


In [20]:
dfCO2Sec = dfCO2.merge(dfCO2Map[['sector', 'var1', 'var2', 'var3', 'var4', 'var5']].drop_duplicates(), on=['sector'], how='left')
dfCO2Sec.head()

,Units,scenario,region,sector,Year,value,GHG,var1,var2,var3,var4,var5
0,MTC,Current-Policy,South Korea,H2 central production,2020,0.002983,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen
1,MTC,Current-Policy,South Korea,H2 central production,2025,0.005725,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen
2,MTC,Current-Policy,South Korea,H2 central production,2030,0.002482,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen
3,MTC,Current-Policy,South Korea,H2 central production,2035,0.005033,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen
4,MTC,Current-Policy,South Korea,H2 wholesale dispensing,2020,0.002867,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen


In [21]:
dfNonCO2_1Sec = dfNonCO2_1.merge(dfNonCO2Map[(dfNonCO2Map['subsector'].isna())][['sector', 'ghg', 'var1', 'var2', 'var3', 'var4', 'var5']].drop_duplicates(), left_on=['sector', 'GHG'], right_on=['sector', 'ghg'], how='left')
dfNonCO2_1Sec.head()

,Units,scenario,region,sector,subsector,GHG,Year,value,ghg,var1,var2,var3,var4,var5
0,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2005,0.253100,HFC125,Emissions|HFC,Emissions|HFC|HFC125,Emissions|F-Gases,NaN,NaN
1,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2010,0.629604,HFC125,Emissions|HFC,Emissions|HFC|HFC125,Emissions|F-Gases,NaN,NaN
2,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2015,0.717899,HFC125,Emissions|HFC,Emissions|HFC|HFC125,Emissions|F-Gases,NaN,NaN
3,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2020,0.796548,HFC125,Emissions|HFC,Emissions|HFC|HFC125,Emissions|F-Gases,NaN,NaN
4,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2025,0.783772,HFC125,Emissions|HFC,Emissions|HFC|HFC125,Emissions|F-Gases,NaN,NaN


In [22]:
dfNonCO2_2Sec = dfNonCO2_2.merge(dfNonCO2Map[['sector', 'subsector', 'ghg', 'var1', 'var2', 'var3', 'var4', 'var5']].drop_duplicates(), left_on=['sector', 'subsector', 'GHG'], right_on=['sector', 'subsector', 'ghg'], how='left')
dfNonCO2_2Sec.head()

,Units,scenario,region,sector,subsector,GHG,Year,value,ghg,var1,var2,var3,var4,var5
0,Tg,Current-Policy,South Korea,urban processes,landfills,CH4,1975,0.013917,CH4,Emissions|CH4,Emissions|CH4|Waste,NaN,NaN,NaN
1,Tg,Current-Policy,South Korea,urban processes,landfills,CH4,1990,0.031836,CH4,Emissions|CH4,Emissions|CH4|Waste,NaN,NaN,NaN
2,Tg,Current-Policy,South Korea,urban processes,landfills,CH4,2005,0.042066,CH4,Emissions|CH4,Emissions|CH4|Waste,NaN,NaN,NaN
3,Tg,Current-Policy,South Korea,urban processes,landfills,CH4,2010,0.021372,CH4,Emissions|CH4,Emissions|CH4|Waste,NaN,NaN,NaN
4,Tg,Current-Policy,South Korea,urban processes,landfills,CH4,2015,0.021774,CH4,Emissions|CH4,Emissions|CH4|Waste,NaN,NaN,NaN


In [23]:
dfNonCO2_1.shape, dfNonCO2_1Sec.shape

((6151, 8), (6151, 14))

In [24]:
dfNonCO2_2.shape, dfNonCO2_2Sec.shape

((90, 8), (90, 14))

In [25]:
dfNonCO2Sec = pd.concat([dfNonCO2_1Sec, dfNonCO2_2Sec])
dfNonCO2Sec

,Units,scenario,region,sector,subsector,GHG,Year,value,ghg,var1,var2,var3,var4,var5
0,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2005,0.253100,HFC125,Emissions|HFC,Emissions|HFC|HFC125,Emissions|F-Gases,NaN,NaN
1,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2010,0.629604,HFC125,Emissions|HFC,Emissions|HFC|HFC125,Emissions|F-Gases,NaN,NaN
2,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2015,0.717899,HFC125,Emissions|HFC,Emissions|HFC|HFC125,Emissions|F-Gases,NaN,NaN
3,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2020,0.796548,HFC125,Emissions|HFC,Emissions|HFC|HFC125,Emissions|F-Gases,NaN,NaN
4,Gg,Current-Policy,South Korea,comm cooling,electricity,HFC125,2025,0.783772,HFC125,Emissions|HFC,Emissions|HFC|HFC125,Emissions|F-Gases,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,Tg,Enhanced-Ambition,South Korea,urban processes,wastewater,N2O,2015,0.003289,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN
86,Tg,Enhanced-Ambition,South Korea,urban processes,wastewater,N2O,2020,0.003340,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN
87,Tg,Enhanced-Ambition,South Korea,urban processes,wastewater,N2O,2025,0.003312,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN
88,Tg,Enhanced-Ambition,South Korea,urban processes,wastewater,N2O,2030,0.002672,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN


In [26]:
dfCO2['sector'].unique()

array(['H2 central production', 'H2 wholesale dispensing',
       'agricultural energy use', 'ammonia', 'backup_electricity',
       'cement', 'chemical energy use', 'chemical feedstocks',
       'comm cooling', 'comm heating', 'comm others',
       'construction energy use', 'construction feedstocks',
       'delivered biomass', 'delivered gas', 'desalinated water',
       'elec_coal (IGCC CCS)', 'elec_coal (conv pul CCS)',
       'elec_gas (CC CCS)', 'electricity', 'gas pipeline',
       'gas processing', 'iron and steel', 'mining energy use',
       'other industrial energy use', 'other industrial feedstocks',
       'process heat cement', 'process heat food processing',
       'process heat paper', 'refined liquids enduse',
       'refined liquids industrial', 'refining', 'resid heating coal',
       'resid heating modern', 'resid others coal', 'resid others modern',
       'trn_aviation_intl', 'trn_freight', 'trn_freight_road', 'trn_pass',
       'trn_pass_road', 'trn_pass_road_LD

In [27]:
def cat_sec(row):
    if row['sector'] == 'electricity':
        return 'Electricity'
    elif row['sector'] == 'agricultural energy use':
        return 'Agriculture'
    elif row['var2'].endswith('Other Capture and Removal'):
        return 'DAC'
    elif row['sector'] == 'cement':
        return 'Industry'
    elif row['sector'] == 'desalinated water':
        return 'Buildings'
    # elif row['var5'].endswith('Hydrogen'):
    #     return 'Hydrogen'
    elif row['var5'].endswith('Industry'):
        return 'Industry'
    elif row['var5'].endswith('Electricity'):
        return "Electricity"
    elif row['var5'].endswith('Residential and Commercial'):
        return "Buildings"
    elif row['var5'].endswith('Transportation'):
        return 'Transportation'
    elif row['sector'] in ['delivered biomass', 'delivered gas', 'gas pipeline', 'gas processing', 'refined liquids enduse', 'refined liquids industrial', 'refining', 'wholesale gas']:
        return 'Industry'
    elif row['var5'].endswith('AFOFI'):
        return 'Industry'
    else:
        print(row['sector'])
        return "Others"

In [28]:
dfCO2Sec['sec'] = dfCO2Sec.apply(cat_sec, axis=1)

H2 central production
H2 central production
H2 central production
H2 central production
H2 wholesale dispensing
H2 wholesale dispensing
H2 wholesale dispensing
H2 wholesale dispensing
H2 central production
H2 central production
H2 central production
H2 central production
H2 wholesale dispensing
H2 wholesale dispensing
H2 wholesale dispensing
H2 wholesale dispensing


In [29]:
dfNonCO2Sec[~(dfNonCO2Sec['var4'].isna())]['sector'].unique()

array(['industrial processes', 'Beef', 'Corn', 'Dairy', 'FiberCrop',
       'Fruits', 'H2 central production', 'Legumes', 'MiscCrop',
       'NutsSeeds', 'OilCrop', 'OtherGrain', 'Pork', 'Poultry', 'Rice',
       'RootTuber', 'SheepGoat', 'Soybean', 'UnmanagedLand', 'Vegetables',
       'Wheat', 'agricultural energy use', 'ammonia',
       'backup_electricity', 'biomass', 'chemical energy use',
       'comm cooling', 'comm heating', 'comm others',
       'construction energy use', 'electricity', 'mining energy use',
       'other industrial energy use', 'process heat cement',
       'process heat food processing', 'process heat paper', 'refining',
       'resid heating TradBio', 'resid heating coal',
       'resid heating modern', 'resid others TradBio',
       'resid others coal', 'resid others modern', 'trn_aviation_intl',
       'trn_freight', 'trn_freight_road', 'trn_pass', 'trn_pass_road',
       'trn_pass_road_LDV', 'trn_pass_road_LDV_4W', 'trn_shipping_intl'],
      dtype=object

In [30]:
dfNonCO2Sec[(dfNonCO2Sec['sector'] == 'urban processes') & ~(dfNonCO2Sec['GHG'].isin(['HFC125', 'HFC134a', 'HFC143a', 'HFC23', 'HFC32', 'HFC43', 'HFC227ea', 'HFC236fa', 'SF6', 'C2F6', 'CF4']))]['var2'].unique()

array(['Emissions|CH4|Waste', 'Emissions|N2O|Waste'], dtype=object)

In [31]:
dfNonCO2Sec['sector'].unique()

array(['comm cooling', 'electricity_net_ownuse', 'industrial processes',
       'resid cooling modern', 'urban processes', 'Beef', 'Corn', 'Dairy',
       'FiberCrop', 'FodderGrass', 'Fruits', 'H2 central production',
       'Legumes', 'MiscCrop', 'NutsSeeds', 'OilCrop', 'OtherGrain',
       'Pork', 'Poultry', 'Rice', 'RootTuber', 'SheepGoat', 'Soybean',
       'UnmanagedLand', 'Vegetables', 'Wheat', 'agricultural energy use',
       'ammonia', 'backup_electricity', 'biomass', 'chemical energy use',
       'comm heating', 'comm others', 'construction energy use',
       'electricity', 'iron and steel', 'mining energy use',
       'other industrial energy use', 'process heat cement',
       'process heat food processing', 'process heat paper', 'refining',
       'resid heating TradBio', 'resid heating coal',
       'resid heating modern', 'resid others TradBio',
       'resid others coal', 'resid others modern', 'trn_aviation_intl',
       'trn_freight', 'trn_freight_road', 'trn_pass', 

In [32]:
def cat_sec_nonco2(row):
    if row['GHG'] in ['HFC125', 'HFC134a', 'HFC143a', 'HFC23', 'HFC32', 'HFC43', 'HFC227ea', 'HFC236fa', 'SF6', 'C2F6', 'CF4']:
        return 'F-Gases'
    elif row['GHG'] in ['CH4', 'CH4_AWB', 'CH4_AGR']:
        return 'Methane'
    elif row['sector'] == 'agricultural energy use':
        return "Agriculture"
    elif row['var2'].endswith("Waste"):
        return "Waste"
    elif (row['var2'].endswith("AFOLU")) and ((row['var3'].endswith("Agriculture")) or (row['var3'].endswith('Agricultural Waste Burning'))):
        return 'Agriculture'
    elif (row['var2'].endswith("AFOLU")):
        return 'Others'
    elif (row['var2'].endswith('Industrial Processes')) or (row['sector'] == 'industrial processes'):
        return 'Industry'
    elif (row['var2'].endswith('Waste')) or (row['sector'] == 'urban processes'):
        return 'Others'#'Waste'
    elif row['var4'].endswith('Electricity'):
        return 'Electricity'
    elif (row['var2'].endswith('Industrial Processes')) or (row['var4'].endswith('Industry')):
        return 'Industry'
    elif row['var4'].endswith('Transportation'):
        return 'Transportation'
    # elif row['var4'].endswith('Hydrogen'):
    #     return 'Hydrogen' 
    elif row['var4'].endswith('Residential and Commercial'):
        return "Buildings"
    elif (row['var4'].endswith('AFOFI')) or (row['var4'].endswith('Heat')) or (row['var4'].endswith('Liquids')):
        return 'Industry'
    else:
        print(row['sector'])
        return "Others"

In [33]:
dfNonCO2Sec['sec'] = dfNonCO2Sec.apply(cat_sec_nonco2, axis=1)

trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_aviation_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl
trn_shipping_intl


In [34]:
dfGHGSec = pd.concat([dfCO2Sec, dfNonCO2Sec])
dfGHGSec

,Units,scenario,region,sector,Year,value,GHG,var1,var2,var3,var4,var5,sec,subsector,ghg
0,MTC,Current-Policy,South Korea,H2 central production,2020,0.002983,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN
1,MTC,Current-Policy,South Korea,H2 central production,2025,0.005725,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN
2,MTC,Current-Policy,South Korea,H2 central production,2030,0.002482,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN
3,MTC,Current-Policy,South Korea,H2 central production,2035,0.005033,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN
4,MTC,Current-Policy,South Korea,H2 wholesale dispensing,2020,0.002867,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,Tg,Enhanced-Ambition,South Korea,urban processes,2015,0.003289,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN,Waste,wastewater,N2O
86,Tg,Enhanced-Ambition,South Korea,urban processes,2020,0.003340,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN,Waste,wastewater,N2O
87,Tg,Enhanced-Ambition,South Korea,urban processes,2025,0.003312,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN,Waste,wastewater,N2O
88,Tg,Enhanced-Ambition,South Korea,urban processes,2030,0.002672,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN,Waste,wastewater,N2O


In [35]:
dfGHGSec['sec'].unique()

array(['Others', 'Agriculture', 'Industry', 'Electricity', 'Buildings',
       'Transportation', 'DAC', 'F-Gases', 'Methane', 'Waste'],
      dtype=object)

In [36]:
dfGHGSec[(dfGHGSec['sec'] == 'Agriculture')]['var3'].unique()

array(['Emissions|CO2|Energy', 'Emissions|N2O|AFOLU|Agriculture',
       'Emissions|N2O|AFOLU|Agricultural Waste Burning',
       'Emissions|N2O|Energy|Demand'], dtype=object)

In [37]:
dfGHGSec['var3'].unique()

array(['Emissions|CO2|Energy', 'Emissions|CO2|Industrial Processes', nan,
       'Emissions|F-Gases', 'Emissions|CH4|AFOLU|Agriculture',
       'Emissions|N2O|AFOLU|Agriculture',
       'Emissions|CH4|AFOLU|Agricultural Waste Burning',
       'Emissions|N2O|AFOLU|Agricultural Waste Burning',
       'Emissions|N2O|AFOLU|Land', 'Emissions|CH4|Energy|Supply',
       'Emissions|CH4|AFOLU|Land', 'Emissions|BC|Energy|Demand',
       'Emissions|N2O|Energy|Demand',
       'Emissions|CH4|Industrial Processes|Chemicals',
       'Emissions|N2O|Industrial Processes|Chemicals',
       'Emissions|N2O|Energy|Supply', 'Emissions|CH4|Energy|Demand',
       'Emissions|CH4|Industrial Processes|Iron and Steel',
       'Emissions|N2O|Industrial Processes|Iron and Steel',
       'Emissions|CH4|Industrial Processes|Pulp and Paper',
       'Emissions|N2O|Industrial Processes|Pulp and Paper'], dtype=object)

In [38]:
dfGHGSec['emiss(MT)'] = dfGHGSec.apply(convert_to_mt, axis=1)
dfGHGSec['gwpAr5'] = dfGHGSec['GHG'].apply(lambda gas: GWP_AR5[gas] if gas in GWP_AR5 else np.nan)
dfGHGSec['MTCO2eq'] = dfGHGSec['emiss(MT)'] * dfGHGSec['gwpAr5']
dfGHGSec

,Units,scenario,region,sector,Year,value,GHG,var1,var2,var3,var4,var5,sec,subsector,ghg,emiss(MT),gwpAr5,MTCO2eq
0,MTC,Current-Policy,South Korea,H2 central production,2020,0.002983,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,0.010939,1,0.010939
1,MTC,Current-Policy,South Korea,H2 central production,2025,0.005725,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,0.020990,1,0.020990
2,MTC,Current-Policy,South Korea,H2 central production,2030,0.002482,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,0.009102,1,0.009102
3,MTC,Current-Policy,South Korea,H2 central production,2035,0.005033,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,0.018453,1,0.018453
4,MTC,Current-Policy,South Korea,H2 wholesale dispensing,2020,0.002867,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,0.010512,1,0.010512
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
85,Tg,Enhanced-Ambition,South Korea,urban processes,2015,0.003289,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN,Waste,wastewater,N2O,0.003289,265,0.871601
86,Tg,Enhanced-Ambition,South Korea,urban processes,2020,0.003340,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN,Waste,wastewater,N2O,0.003340,265,0.885222
87,Tg,Enhanced-Ambition,South Korea,urban processes,2025,0.003312,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN,Waste,wastewater,N2O,0.003312,265,0.877770
88,Tg,Enhanced-Ambition,South Korea,urban processes,2030,0.002672,N2O,Emissions|N2O,Emissions|N2O|Waste,NaN,NaN,NaN,Waste,wastewater,N2O,0.002672,265,0.708146


In [39]:
dfGHGSec[(dfGHGSec['sector'] == 'Rice') & (dfGHGSec['Year'] == 2035)].groupby(['scenario', 'Year', 'subsector'])['MTCO2eq'].sum()

scenario           Year  subsector 
Current-Policy     2035  Rice_Korea    4.533576
Enhanced-Ambition  2035  Rice_Korea    3.769450
Name: MTCO2eq, dtype: float64

In [40]:
dfGHGSec[(dfGHGSec['sec'].isin(['Agriculture']))& (dfGHGSec['Year'] == 2035)].groupby(['scenario', 'Year', 'sector'])['MTCO2eq'].sum()

scenario           Year  sector                 
Current-Policy     2035  Beef                       1.083008
                         Corn                       0.110850
                         Dairy                      0.419750
                         FiberCrop                  0.001846
                         Fruits                     0.473291
                         Legumes                    0.056300
                         MiscCrop                   0.065226
                         NutsSeeds                  0.147892
                         OilCrop                    0.345258
                         OtherGrain                 0.170423
                         Pork                       1.028910
                         Poultry                    0.056277
                         Rice                       0.003571
                         RootTuber                  0.152935
                         SheepGoat                  0.060928
                         Soybean    

In [41]:
dfGHGSec[(dfGHGSec['Year'].isin([2020, 2035])) & (dfGHGSec['sec'] == 'Waste')].groupby(['scenario', 'Year', 'subsector'])['MTCO2eq'].sum()

scenario           Year  subsector         
Current-Policy     2020  waste_incineration    0.010686
                         wastewater            0.885222
                   2035  waste_incineration    0.009113
                         wastewater            0.752131
Enhanced-Ambition  2020  waste_incineration    0.010686
                         wastewater            0.885222
                   2035  waste_incineration    0.007582
                         wastewater            0.625893
Name: MTCO2eq, dtype: float64

In [42]:
dfGHGSec.groupby(['scenario', 'Year'])['MTCO2eq'].sum()

scenario           Year
Current-Policy     1975     54.168502
                   1990    296.523004
                   2005    601.039565
                   2010    698.621279
                   2015    747.006066
                   2020    737.535822
                   2025    707.932348
                   2030    633.027329
                   2035    568.177049
Enhanced-Ambition  1975     54.168502
                   1990    296.523004
                   2005    601.039565
                   2010    698.621279
                   2015    747.006066
                   2020    736.316569
                   2025    707.112481
                   2030    534.660076
                   2035    390.427011
Name: MTCO2eq, dtype: float64

In [43]:
dfGHGSec[(dfGHGSec['sector'].str.contains('H2')) & (~dfGHGSec['sector'].isin(['trn_aviation_intl', 'trn_shipping_intl'])) & (dfGHGSec['Year'] == 2035)]#['var4'].unique()

,Units,scenario,region,sector,Year,value,GHG,var1,var2,var3,var4,var5,sec,subsector,ghg,emiss(MT),gwpAr5,MTCO2eq
3,MTC,Current-Policy,South Korea,H2 central production,2035,5.032685e-03,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,1.845318e-02,1,0.018453
7,MTC,Current-Policy,South Korea,H2 wholesale dispensing,2035,5.475123e-02,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,2.007545e-01,1,0.200755
597,MTC,Enhanced-Ambition,South Korea,H2 central production,2035,8.399805e-04,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,3.079928e-03,1,0.003080
601,MTC,Enhanced-Ambition,South Korea,H2 wholesale dispensing,2035,1.861793e-03,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,6.826573e-03,1,0.006827
1096,Tg,Current-Policy,South Korea,H2 central production,2035,2.437059e-06,CH4,Emissions|CH4,Emissions|CH4|Energy,Emissions|CH4|Energy|Supply,Emissions|CH4|Energy|Supply|Hydrogen,NaN,Methane,biomass,CH4,2.437059e-06,28,0.000068
3704,Tg,Enhanced-Ambition,South Korea,H2 central production,2035,1.822670e-07,CH4,Emissions|CH4,Emissions|CH4|Energy,Emissions|CH4|Energy|Supply,Emissions|CH4|Energy|Supply|Hydrogen,NaN,Methane,biomass,CH4,1.822670e-07,28,0.000005
3707,Tg,Enhanced-Ambition,South Korea,H2 central production,2035,2.934120e-07,CH4,Emissions|CH4,Emissions|CH4|Energy,Emissions|CH4|Energy|Supply,Emissions|CH4|Energy|Supply|Hydrogen,NaN,Methane,coal,CH4,2.934120e-07,28,0.000008


In [44]:
dfGHGSec[(dfGHGSec['sec'] == 'Others') & (~dfGHGSec['sector'].isin(['trn_aviation_intl', 'trn_shipping_intl'])) & (dfGHGSec['Year'] == 2035)]#['var4'].unique()

,Units,scenario,region,sector,Year,value,GHG,var1,var2,var3,var4,var5,sec,subsector,ghg,emiss(MT),gwpAr5,MTCO2eq
3,MTC,Current-Policy,South Korea,H2 central production,2035,0.005033,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,0.018453,1,0.018453
7,MTC,Current-Policy,South Korea,H2 wholesale dispensing,2035,0.054751,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,0.200755,1,0.200755
597,MTC,Enhanced-Ambition,South Korea,H2 central production,2035,0.000840,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,0.003080,1,0.003080
601,MTC,Enhanced-Ambition,South Korea,H2 wholesale dispensing,2035,0.001862,CO2,Emissions|CO2,Emissions|CO2|Energy and Industrial Processes,Emissions|CO2|Energy,Emissions|CO2|Energy|Supply,Emissions|CO2|Energy|Supply|Hydrogen,Others,NaN,NaN,0.006827,1,0.006827
1038,Tg,Current-Policy,South Korea,FodderGrass,2035,0.000120,N2O_AGR,Emissions|N2O,Emissions|N2O|AFOLU,Emissions|N2O|AFOLU|Land,NaN,NaN,Others,FodderGrass_Korea,N2O_AGR,0.000120,265,0.031847
1453,Tg,Current-Policy,South Korea,UnmanagedLand,2035,0.000136,N2O,Emissions|N2O,Emissions|N2O|AFOLU,Emissions|N2O|AFOLU|Land,Emissions|N2O|AFOLU|Land|Other,NaN,Others,Deforest_Korea,N2O,0.000136,265,0.035994
1471,Tg,Current-Policy,South Korea,UnmanagedLand,2035,0.000004,N2O,Emissions|N2O,Emissions|N2O|AFOLU,Emissions|N2O|AFOLU|Land,Emissions|N2O|AFOLU|Land|Other,NaN,Others,ForestFire_Korea,N2O,0.000004,265,0.001177
1489,Tg,Current-Policy,South Korea,UnmanagedLand,2035,0.000013,N2O,Emissions|N2O,Emissions|N2O|AFOLU,Emissions|N2O|AFOLU|Land,Emissions|N2O|AFOLU|Land|Other,NaN,Others,GrasslandFires_Korea,N2O,0.000013,265,0.003464
3646,Tg,Enhanced-Ambition,South Korea,FodderGrass,2035,0.000118,N2O_AGR,Emissions|N2O,Emissions|N2O|AFOLU,Emissions|N2O|AFOLU|Land,NaN,NaN,Others,FodderGrass_Korea,N2O_AGR,0.000118,265,0.031320
4064,Tg,Enhanced-Ambition,South Korea,UnmanagedLand,2035,0.000181,N2O,Emissions|N2O,Emissions|N2O|AFOLU,Emissions|N2O|AFOLU|Land,Emissions|N2O|AFOLU|Land|Other,NaN,Others,Deforest_Korea,N2O,0.000181,265,0.048066


In [45]:
dfGHGSec[(dfGHGSec['sec'] == 'Industry')]['sector'].unique()

array(['ammonia', 'cement', 'chemical energy use', 'chemical feedstocks',
       'construction energy use', 'construction feedstocks',
       'delivered biomass', 'delivered gas', 'gas pipeline',
       'gas processing', 'iron and steel', 'mining energy use',
       'other industrial energy use', 'other industrial feedstocks',
       'process heat cement', 'process heat food processing',
       'process heat paper', 'refined liquids enduse',
       'refined liquids industrial', 'refining',
       'waste biomass for paper', 'wholesale gas', 'process heat dac',
       'industrial processes'], dtype=object)

In [46]:
dfGHGSec[(dfGHGSec['sector'].isin(['cement', 'chemical energy use', 'chemical feedstocks', 'iron and steel']))].groupby(['scenario', 'Year'])['MTCO2eq'].sum()

scenario           Year
Current-Policy     1975      7.326476
                   1990     48.589054
                   2005     35.000848
                   2010     37.239279
                   2015    120.490139
                   2020    115.982553
                   2025    108.872008
                   2030    107.708864
                   2035     94.495743
Enhanced-Ambition  1975      7.326476
                   1990     48.589054
                   2005     35.000848
                   2010     37.239279
                   2015    120.490139
                   2020    116.000256
                   2025    108.807296
                   2030     67.181148
                   2035     37.032034
Name: MTCO2eq, dtype: float64

In [47]:
dfGHGSec[(dfGHGSec['sector'].isin(['iron and steel']))].groupby(['scenario', 'Year'])['MTCO2eq'].sum()

scenario           Year
Current-Policy     1975     2.273809
                   1990    24.701654
                   2005     3.868226
                   2010     4.430452
                   2015    89.216062
                   2020    88.808010
                   2025    83.365276
                   2030    83.965948
                   2035    72.786746
Enhanced-Ambition  1975     2.273809
                   1990    24.701654
                   2005     3.868226
                   2010     4.430452
                   2015    89.216062
                   2020    88.808474
                   2025    83.293296
                   2030    55.045596
                   2035    31.764331
Name: MTCO2eq, dtype: float64

In [48]:
dfGHGSec[(dfGHGSec['sector'].isin(['chemical energy use', 'chemical feedstocks']))].groupby(['scenario', 'Year'])['MTCO2eq'].sum()

scenario           Year
Current-Policy     1990    7.130734
                   2005    5.506288
                   2010    7.813160
                   2015    6.278410
                   2020    5.135420
                   2025    5.066204
                   2030    5.120507
                   2035    4.816188
Enhanced-Ambition  1990    7.130734
                   2005    5.506288
                   2010    7.813160
                   2015    6.278410
                   2020    5.151963
                   2025    5.082672
                   2030   -4.106136
                   2035   -8.575238
Name: MTCO2eq, dtype: float64

In [49]:
dfGHGSec[(dfGHGSec['sector'].isin(['cement']))].groupby(['scenario', 'Year'])['MTCO2eq'].sum()

scenario           Year
Current-Policy     1975     5.052667
                   1990    16.756667
                   2005    25.626333
                   2010    24.995667
                   2015    24.995667
                   2020    22.039123
                   2025    20.440528
                   2030    18.622409
                   2035    16.892809
Enhanced-Ambition  1975     5.052667
                   1990    16.756667
                   2005    25.626333
                   2010    24.995667
                   2015    24.995667
                   2020    22.039820
                   2025    20.431329
                   2030    16.241688
                   2035    13.842941
Name: MTCO2eq, dtype: float64

In [50]:
dfGHGSec[(dfGHGSec['sec'] == 'Industry') & (~dfGHGSec['sector'].isin(['cement', 'chemical energy use', 'chemical feedstocks', 'iron and steel']))].groupby(['scenario', 'Year'])['MTCO2eq'].sum()

scenario           Year
Current-Policy     1975     22.059122
                   1990     62.871556
                   2005    142.176127
                   2010    157.130497
                   2015    105.075429
                   2020    104.995628
                   2025    113.763509
                   2030    111.936225
                   2035    111.041576
Enhanced-Ambition  1975     22.059122
                   1990     62.871556
                   2005    142.176127
                   2010    157.130497
                   2015    105.075429
                   2020    104.915014
                   2025    114.025558
                   2030    106.644727
                   2035    110.210450
Name: MTCO2eq, dtype: float64

In [51]:
dfGHGSecDiff_ = dfGHGSec[(dfGHGSec['Year'].isin([2020, 2030]) & (~dfGHGSec['sector'].isin(['trn_aviation_intl', 'trn_shipping_intl']))) & (dfGHGSec['scenario'].isin(['Current-Policy', 'Enhanced-Ambition', 'Enhanced-Ambition-Bld']))].groupby(['scenario', 'Year', 'sec'])['MTCO2eq'].sum().reset_index().pivot(index=['scenario', 'sec'], columns=['Year'], values='MTCO2eq')
dfGHGSecDiff_

Year                                    2020        2030
scenario          sec                                   
Current-Policy    Agriculture       7.487551    6.320148
                  Buildings        50.701919   47.780316
                  Electricity     243.863571  161.510830
                  F-Gases          42.535721   37.269381
                  Industry        220.843311  219.453806
                  Methane          27.229002   26.944813
                  Others            0.059192    0.202081
                  Transportation  117.538991  106.156607
                  Waste             0.895908    0.793632
Enhanced-Ambition Agriculture       7.487520    6.248585
                  Buildings        50.697220   43.501906
                  DAC                    NaN   -2.959733
                  Electricity     243.861561  139.348670
                  F-Gases          42.536429   27.482655
                  Industry        220.780401  173.655614
                  Methane          27.211308   24.328249
                  Others            0.059186    0.091524
                  Transportation  116.403434   96.409844
                  Waste             0.895908    0.716717

In [52]:
161.59 - 243.96

-82.37

In [53]:
dfGHGSec[(dfGHGSec['sec'] == 'Industry')]['sector'].unique()

array(['ammonia', 'cement', 'chemical energy use', 'chemical feedstocks',
       'construction energy use', 'construction feedstocks',
       'delivered biomass', 'delivered gas', 'gas pipeline',
       'gas processing', 'iron and steel', 'mining energy use',
       'other industrial energy use', 'other industrial feedstocks',
       'process heat cement', 'process heat food processing',
       'process heat paper', 'refined liquids enduse',
       'refined liquids industrial', 'refining',
       'waste biomass for paper', 'wholesale gas', 'process heat dac',
       'industrial processes'], dtype=object)

In [54]:
def cat_ind_sec(sector):
    if sector in ['iron and steel']:
        return 'Iron and Steel'
    elif sector in ['ammonia', 'chemical energy use', 'chemical feedstocks',]:
        return 'Chemical'
    elif sector in ['cement', 'process heat cement']:
        return 'Cement'
    else:
        return 'Other Industry'

In [55]:
dfGHGSec['sec'].unique()

array(['Others', 'Agriculture', 'Industry', 'Electricity', 'Buildings',
       'Transportation', 'DAC', 'F-Gases', 'Methane', 'Waste'],
      dtype=object)

In [56]:
dfGHGSecInd = dfGHGSec[(dfGHGSec['sec'] == 'Industry')].copy()
dfGHGSecInd['ind_sec'] = dfGHGSecInd['sector'].apply(cat_ind_sec)

In [57]:
dfGHGSecIndDiff = dfGHGSecInd[(dfGHGSecInd['Year'].isin([2020, 2035])) & (dfGHGSecInd['scenario'].isin(['Current-Policy', 'Enhanced-Ambition']))].groupby(['scenario', 'Year', 'ind_sec'])['MTCO2eq'].sum().reset_index().pivot(index=['scenario', 'ind_sec'], columns=['Year'], values='MTCO2eq').reset_index()
dfGHGSecIndDiff.fillna(0, inplace=True)
dfGHGSecIndDiff['Diff'] = dfGHGSecIndDiff[2035] - dfGHGSecIndDiff[2020]
dfGHGSecIndDiff

Year,scenario,ind_sec,2020,2035,Diff
0,Current-Policy,Cement,33.591629,29.346234,-4.245395
1,Current-Policy,Chemical,5.901401,6.203906,0.302506
2,Current-Policy,Iron and Steel,88.715103,72.606401,-16.108702
3,Current-Policy,Other Industry,92.635180,97.159451,4.524271
4,Enhanced-Ambition,Cement,33.592795,24.202121,-9.390674
5,Enhanced-Ambition,Chemical,5.917941,-7.854607,-13.772548
6,Enhanced-Ambition,Iron and Steel,88.715565,31.629275,-57.086290
7,Enhanced-Ambition,Other Industry,92.554100,99.087088,6.532988


In [58]:
dfGHGSecIndDiff[(dfGHGSecIndDiff['scenario'] == 'Enhanced-Ambition')][2020].sum()

np.float64(220.78040080141653)

In [59]:
dfGHGSecIndDiff[(dfGHGSecIndDiff['scenario'] == 'Enhanced-Ambition')][2035].sum()

np.float64(147.06387713673058)

In [60]:
dfGHGSecIndDiff[(dfGHGSecIndDiff['scenario'] == 'Enhanced-Ambition')]['Diff'].sum()

np.float64(-73.71652366468595)

In [61]:
-73.71 -7.07-2.43-2.49-0.06 + 89.8

4.039999999999992

In [62]:
emiss_2018 = 269.98

steel_diff = 7.06
chem_diff = 2.35
cement_diff = 2.50
other_diff = 0.07

steel_2018 = 111.48
chem_2018 = 54.84
cement_2018 = 42.63
other_2018 = 93.92

steel_base = emiss_2018
steel_ep = -57.08 - steel_diff 
steel_cp = -16.10 - steel_diff 
chem_base = emiss_2018 + steel_ep
chem_ep = -13.77 - chem_diff
chem_cp = 0 - chem_diff
cement_base = chem_base + chem_ep
cement_ep = -9.39 - cement_diff
cement_cp = -4.24 - cement_diff
other_base = cement_base + cement_ep
other_ep = -4.04 - other_diff
other_cp = -4.04 - other_diff

emiss_2035_ep = emiss_2018 - 89.8

rr_steel =  - (steel_ep / steel_2018) * 100
rr_chem = - (chem_ep / chem_2018) * 100
rr_cement = - (cement_ep / cement_2018) * 100
rr_other = - (other_ep / other_2018) * 100

In [63]:
rd_ttl = emiss_2035_ep - emiss_2018
rr_ttl = -rd_ttl / emiss_2018 * 100

In [64]:
rr_ttl

33.261723090599304

In [ ]:
data = pd.DataFrame({
    "category": [
        "2018",
        "Iron & Steel", "Chemical", "Cement", "Other",
        "2035"
    ],
})

fig = go.Figure()

# Start and end bars
fig.add_trace(go.Waterfall(
    name="Enhanced Ambition",
    orientation="v",
    measure=["absolute"] * 1 + ["relative"] * 4 + ["total"],
    x=data["category"],
    y=[emiss_2018, steel_ep, chem_ep, cement_ep, other_ep, emiss_2035_ep],
    base=0,
    connector={"visible": False},
    decreasing={"marker": {"color": "#1f77b4"}},   # enhanced ambition
    increasing={"marker": {"color": "#FF6692"}},   # current policies
    totals={"marker": {"color": "lightgray"}},
    showlegend=False
))
pio.write_image(fig, "./fig/emiss_by_industry_sector.png", width=650, height=680, scale=2)
fig

In [67]:
data = pd.DataFrame({
    "category": [
        "2018",
        "Iron & Steel", "Chemical", "Cement", "Other Industry",
        "2035"
    ],
})

fig = go.Figure()

# Start and end bars
fig.add_trace(go.Waterfall(
    name="Enhanced Ambition",
    orientation="v",
    measure=["absolute"] * 1 + ["relative"] * 4 + ["total"],
    x=data["category"],
    y=[emiss_2018, steel_ep, chem_ep, cement_ep, other_ep, emiss_2035_ep],
    base=0,
    connector={"visible": False},
    decreasing={"marker": {"color": "#1f77b4"}},   # enhanced ambition
    increasing={"marker": {"color": "#FF6692"}},   # current policies
    totals={"marker": {"color": "lightgray"}},
    showlegend=False
))

# Add Current Policy overlays just for Coal categories
fig.add_trace(go.Bar(
    name="Current Policy",
    x=["Iron & Steel", "Chemical", "Cement", "Other Industry"],
    y=[steel_cp, chem_cp, cement_cp, other_cp],  # smaller reductions
    base=[steel_base, chem_base, cement_base, other_base],  # position on top of previous waterfall step
    marker_color="#AEC7E8",
    showlegend=False
))

fig.update_layout(
    width=600,
    height=500,
    font=dict(size=12),
    plot_bgcolor='white',
    paper_bgcolor='white',


)

# Add dummy scatter trace to label bar values at center
fig.add_trace(go.Scatter(
    x=["2018", "2035"],
    y=[emiss_2018 / 2, emiss_2035_ep / 2],
    mode="text",
    text=[f"<b>{emiss_2018:.1f}</b>", f"<b>{emiss_2035_ep:.1f}</b>"],
    textposition="middle center",
    showlegend=False
))

# Add dummy scatter trace to label bar values at center
fig.add_trace(go.Scatter(
    x=["Iron & Steel", "Chemical", "Cement", "Other Industry"],
    #-190.5, -73.8, -28.9, -15.9, -4.5, -4.9, -3.0, -9
    y=[x+20 for x in [steel_base, chem_base, cement_base, other_base]],
    mode="text",
    text=[f"<b>{steel_ep:.1f}<br>(△{rr_steel:.1f}%)</b>", f"<b>{chem_ep:.1f}<br>(△{rr_chem:.1f}%)</b>", f"<b>{cement_ep:.1f}<br>(△{rr_cement:.1f}%)</b>", 
          f"<b>{other_ep:.1f}<br>(△{rr_other:.1f}%)</b>"],
    textposition="middle center",
    showlegend=False
))

fig.add_trace(go.Bar(
    x=[None], y=[None],
    name="Current Policy",
    marker=dict(color="#AEC7E8"),
    showlegend=True,
    hoverinfo="skip"
))

fig.add_trace(go.Bar(
    x=[None], y=[None],
    name="Enhanced Ambition",
    marker=dict(color="#1f77b4"),
    showlegend=True,
    hoverinfo="skip"
))


fig.update_layout(
    legend=dict(
        orientation='h',
        yanchor='bottom',
        y=-0.3,
        xanchor='center',
        x=0.5,
        bgcolor='rgba(0,0,0,0)',
        borderwidth=0,
        font=dict(size=15)
    )
)

fig.update_layout(
    yaxis=dict(
        showgrid=True,
        gridcolor='lightgrey',
        title="Emission (MtCO2e)", title_font_size=18,
        tickvals=list(range(0,351, 50)),
    )
)

fig.add_annotation(
    x=5, y=emiss_2035_ep + 5,        # 7 is the index of "2035" in the x-category list
    ax=5, ay=emiss_2018 + 30,
    xref="x", yref="y",
    axref="x", ayref="y",
    text=f"<b>{rd_ttl:.1f}<br>(△{rr_ttl:.1f}%)</b>",
    showarrow=True,
    arrowhead=3,
    arrowwidth=3,
    arrowsize=1,
    arrowcolor="#1f77b4",
    font=dict(size=12, color="black"),
    align="center"
)

fig.add_annotation(
    x=1 + 0.2, y=chem_base,        # 7 is the index of "2035" in the x-category list
    ax=1 + 0.2, ay=emiss_2018 + 3,
    xref="x", yref="y",
    axref="x", ayref="y",
    # text="<b>Enhanced<br>Ambition</b>",
    showarrow=True,
    arrowhead=1,
    arrowwidth=2,
    arrowsize=1,
    arrowcolor="#00A08B",
    font=dict(size=10, color="#00A08B"),
    align="center"
)

fig.add_annotation(
    x=1 -0.2, y=emiss_2018 + steel_cp - 2,        # 7 is the index of "2035" in the x-category list
    ax=1 - 0.2, ay=emiss_2018 + 3,
    xref="x", yref="y",
    axref="x", ayref="y",
    # text="<b>CoalOut</b>",
    showarrow=True,
    arrowhead=1,
    arrowwidth=2,
    arrowsize=1,
    arrowcolor="#1616A7",
    font=dict(size=10, color="#1616A7"),
    align="center"
)


fig.update_layout(
    xaxis=dict(
        tickvals=list(range(len(data['category']))),
        ticktext=[f"<b>{cat}</b>" for cat in data['category']],
        # tickfont=dict(size=15)
    )
)

fig.add_annotation(
    text="<b>Current<br>Policy</b>",
    # xref="paper", yref="paper",
    x=1-0.5, y=259,
    showarrow=False,
    font=dict(size=8, color="#1616A7"),
    align="left"
)

fig.add_annotation(
    text="<b>Enhanced<br>Ambition</b>",
    # xref="paper", yref="paper",
    x=1+0.6, y=259,
    showarrow=False,
    font=dict(size=8, color="#00A08B"),
    align="left"
)
pio.write_image(fig, "./fig/emiss_by_industry_sector.png", width=600, height=500, scale=2)

fig

In [294]:
318.83 - 315.4

3.430000000000007